In [1]:
import os
import glob
import pandas as pd
import shutil
from tqdm import tqdm
from pathlib import Path
import shutil

In [7]:
def migrate_alps_subjects(
    src_root,
    dst_root,
    exclude_cti_prefixes=("CTI_Dxx", "CTI_Dyy", "CTI_Dzz"),
    verbose=True,
    print_names=True,
    use_tqdm=True,
    tqdm_level=1,
):
    # prepare(tqdm)
    tqdmen = None
    if use_tqdm:
        try:
            from tqdm.auto import tqdm as tqdmen  # Notebook/Console optimisation, auto
        except Exception:
            if verbose:
                print("[INFO] NO PACKAGE 'tqdm' (Use general setting)")
            tqdmen = None

    SRC = Path(src_root)
    DST = Path(dst_root)
    if not SRC.exists():
        raise FileNotFoundError(f"!!!! NO SOURCE DIR: {SRC}")
    DST.mkdir(parents=True, exist_ok=True)

    EXCLUDE_TOP_LEVEL = {"results", "logfile.txt"}  # case-insensitive check

    def log(msg):
        if verbose:
            print(msg)

    def copytree_merge(src: Path, dst: Path):
        shutil.copytree(src, dst, dirs_exist_ok=True)

    def copy_subject(src_subject: Path, dst_subject: Path):
        # COPY ORIGINAL INDIVIDUALS (except: Results / logFile.txt)
        dst_subject.mkdir(parents=True, exist_ok=True)
        for item in src_subject.iterdir():
            if item.name.lower() in EXCLUDE_TOP_LEVEL:
                continue
            target = dst_subject / item.name
            if item.is_dir():
                copytree_merge(item, target)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(item, target)

        # CREATE DESTINATION: Results/CTI
        dst_cti = dst_subject / "Results" / "CTI"
        dst_cti.mkdir(parents=True, exist_ok=True)

        # ORIGINAL Results/CTI --> EACH DESTINATION (except: CTI_Dxx/Dyy/Dzz*)
        src_cti = src_subject / "Results" / "CTI"
        if src_cti.exists() and src_cti.is_dir():
            entries = list(src_cti.iterdir())
            iter_entries = entries
            inner_bar = None
            if tqdmen and tqdm_level >= 2:
                inner_bar = tqdmen(iter_entries, desc=f"CTI copy: {src_subject.name}", leave=False)
                iter_entries = inner_bar
            try:
                for entry in iter_entries:
                    if entry.name.startswith(exclude_cti_prefixes):
                        continue
                    if entry.is_dir():
                        copytree_merge(entry, dst_cti / entry.name)
                    else:
                        shutil.copy2(entry, dst_cti / entry.name)
            finally:
                if inner_bar is not None:
                    inner_bar.close()

    def cleanup_subject(src_subject: Path):
        for item in src_subject.iterdir():
            if item.name.lower() in EXCLUDE_TOP_LEVEL:
                continue
            try:
                if item.is_dir():
                    shutil.rmtree(item)
                else:
                    item.unlink(missing_ok=True)  # py3.8 이하면 try/except로 대체
            except Exception as e:
                log(f"[WARN] cleanup failed: {src_subject.name} -> {item.name}: {e}")

    subjects = [p for p in SRC.iterdir() if p.is_dir()]
    ok_names, fail = [], []

    iterator = subjects
    outer_bar = None
    if tqdmen:
        outer_bar = tqdmen(subjects, total=len(subjects), desc="Migrating subjects", leave=True)
        iterator = outer_bar

    try:
        for subj in iterator:
            try:
                # 1) COPY
                copy_subject(subj, DST / subj.name)
                # 2) CLEAN UP INNER FORDER of SUCCESS case (except: Results / logFile.txt)
                cleanup_subject(subj)

                ok_names.append(subj.name)
                if not tqdmen:
                    log(f"[OK] {subj.name}")
            except Exception as e:
                fail.append((subj.name, str(e)))
                if not tqdmen:
                    log(f"[FAIL] {subj.name} -> {e}")
    finally:
        if outer_bar is not None:
            outer_bar.close()

    # print(SUMMARY)
    if verbose:
        print(f"\n[SUMMARY] Success: {len(ok_names)}, Fail: {len(fail)}, Sum n: {len(subjects)}")
        if print_names:
            print(f"[MOVED COUNT] {len(ok_names)}")
            if ok_names:
                print("[MOVED NAMES]")
                for name in ok_names:
                    print(f" - {name}")

    return {
        "subjects_total": len(subjects),
        "subjects_ok": len(ok_names),
        "subjects_fail": len(fail),
        "moved_names": ok_names,
        "failed": fail,
    }

In [8]:
# tst = migrate_alps_subjects(
#     r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\SPEC",
#     r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
#     , use_tqdm=True
#     , tqdm_level=1
# )
# print(tst["subjects_ok"], tst["moved_names"])

Migrating subjects: 100%|████████████████████████████████████████████████████████████████| 5/5 [00:01<00:00,  4.29it/s]


[SUMMARY] Success: 5, Fail: 0, Sum n: 5
[MOVED COUNT] 5
[MOVED NAMES]
 - MWI_CN01_KNH_NODDI32
 - MWI_CN04_SSS_NODDI32
 - MWI_CN05_LSS_NODDI32
 - MWI_CN06_KDS_NODDI32
 - MWI_CN07_KSB_NODDI32
5 ['MWI_CN01_KNH_NODDI32', 'MWI_CN04_SSS_NODDI32', 'MWI_CN05_LSS_NODDI32', 'MWI_CN06_KDS_NODDI32', 'MWI_CN07_KSB_NODDI32']


In [ ]:
# dki = migrate_alps_subjects(
#     r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\DKI",
#     r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
#     , use_tqdm=True
#     , tqdm_level=1
# )
# print(dki["subjects_ok"], dki["moved_names"])

In [9]:
cnmci = migrate_alps_subjects(
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\1_8641_CN_MCI",
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
    , use_tqdm=True
    , tqdm_level=1
)
print(cnmci["subjects_ok"], cnmci["moved_names"])

Migrating subjects: 100%|██████████████████████████████████████████████████████████████| 59/59 [06:33<00:00,  6.67s/it]


[SUMMARY] Success: 59, Fail: 0, Sum n: 59
[MOVED COUNT] 59
[MOVED NAMES]
 - Bin
 - MWI_CN01_KNH_NODDI32
 - MWI_CN04_SSS_NODDI32
 - MWI_CN05_LSS_NODDI32
 - MWI_CN06_KDS_NODDI32
 - MWI_CN07_KSB_NODDI32
 - MWI_CN09_YKJ_NODDI32
 - MWI_CN10_CJS_NODDI32
 - MWI_CN12_LJK_NODDI32
 - MWI_CN13_KH_NODDI32
 - MWI_CN14_PYJ_NODDI32
 - MWI_CN15_YCJ_NODDI32
 - MWI_CN16_LCS_NODDI32
 - MWI_CN17_LPS_NODDI32
 - MWI_CN18_HCS_NODDI32
 - MWI_CN19_CSO_NODDI32
 - MWI_CN20_PYJ_NODDI32
 - MWI_CN21_PYB_NODDI32
 - MWI_CN22_YYJ_NODDI32
 - MWI_CN23_HJL_NODDI32
 - MWI_CN24_PKJ_NODDI32
 - MWI_CN25_NKJ_NODDI32
 - MWI_CN26_LSR_NODDI32
 - MWI_CN27_LSJ_NODDI32
 - MWI_CN28_KOS_NODDI32
 - MWI_CN29_KSH_NODDI32
 - MWI_CN30_PYS_NODDI32
 - MWI_CN31_LOH_NODDI32
 - MWI_CN32_KYH_NODDI32
 - MWI_CN33_LSH_NODDI32
 - MWI_CN34_OJS_NODDI32
 - MWI_CN35_LES_NODDI32
 - MWI_CN36_KER_NODDI32
 - MWI_CN37_KJR_NODDI32
 - MWI_CN38_JJB_NODDI32
 - MWI_CN39_LTS_NODDI32
 - MWI_CN40_LDS_NODDI32
 - MWI_CN41_JHL_NODDI32
 - MWI_CN57_YHY_NODDI32
 - MWI_C

In [10]:
dem19 = migrate_alps_subjects(
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2019\prep",
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
    , use_tqdm=True
    , tqdm_level=1
)
print(dem19["subjects_ok"], dem19["moved_names"])

Migrating subjects: 100%|██████████████████████████████████████████████████████████████| 70/70 [10:32<00:00,  9.03s/it]


[SUMMARY] Success: 70, Fail: 0, Sum n: 70
[MOVED COUNT] 70
[MOVED NAMES]
 - ADEPT01_YSJ_NODDI24
 - ADEPT02_LKY_NODDI24
 - ADEPT03_YKJ_NODDI24
 - ADEPT04_JWS_NODDI24
 - ADEPT05_CBK_NODDI24
 - ADEPT06_PHJ_NODDI24
 - ADEPT07_JG_NODDI24
 - ADEPT08_KMJ_NODDI24
 - ADEPT09_JCJ_NODDI24
 - ADEPT10_YPL_NODDI24
 - ADEPT16_LJS_NODDI24
 - ADEPT18_PST_NODDI24
 - ADEPT21_CJM_NODDI24
 - ADEPT22_KBS_NODDI24
 - ADEPT23_PYS_NODDI24
 - ADEPT24_KJH_NODDI24
 - ADEPT25_JKR_NODDI24
 - ADEPT26_KYJ_NODDI24
 - ADEPT27_KKS_NODDI24
 - ADEPT28_MSH_NODDI24
 - ADEPT29_LBS_NODDI24
 - ADEPT30_WJS_NODDI24
 - ADEPT31_PTL_NODDI24
 - ADEPT32_KSW_NODDI24
 - ADEPT33_YCY_NODDI24
 - ADEPT34_PDY_NODDI24
 - ADEPT36_JJS_NODDI24
 - ADEPT37_KTY_NODDI24
 - ADEPT38_KWW_NODDI24
 - ADEPT39_KKS_NODDI24
 - ADEPT40_JOS_NODDI24
 - ADEPT41_YBO_NODDI24
 - ADEPT42_PSJ_NODDI24
 - ADEPT43_SSY_NODDI24
 - ADEPT44_WDS_NODDI24
 - ADEPT45_KKJ_NODDI24
 - ADEPT46_LJB_NODDI24
 - ADEPT47_SJH_NODDI24
 - ADEPT48_CJK_NODDI24
 - ADEPT49_LKJ_NODDI24
 - ADEP

In [11]:
dem20 = migrate_alps_subjects(
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2020",
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
    , use_tqdm=True
    , tqdm_level=1
)
print(dem20["subjects_ok"], dem20["moved_names"])

Migrating subjects: 100%|██████████████████████████████████████████████████████████████| 30/30 [02:38<00:00,  5.29s/it]


[SUMMARY] Success: 30, Fail: 0, Sum n: 30
[MOVED COUNT] 30
[MOVED NAMES]
 - HFC2020AD01_KYM_NODDI32
 - HFC2020AD02_KS_NODDI32
 - MREPT2020AD01_CGS_NODDI32
 - MREPT2020AD02_HMJ_NODDI32
 - MREPT2020AD03_KJH_NODDI32
 - MREPT2020AD04_NGR_NODDI32
 - MREPT2020AD05_PJE_NODDI32
 - MREPT2020AD06_JHK_NODDI32
 - MREPT2020AD07_KYG_NODDI32
 - MREPT2020AD08_CYS_NODDI32
 - MREPT2020AD09_PJJ_NODDI32
 - MREPT2020AD10_CYJ_NODDI32
 - MREPT2020AD11_KHJ_NODDI32
 - MREPT2020AD12_KOB_NODDI32
 - MREPT2020AD13_SBS_NODDI32
 - MREPT2020AD14_CIL_NODDI32
 - MREPT2020AD15_LHS_NODDI32
 - MREPT2020AD16_CKL_NODDI32
 - MREPT2020AD17_LSH_NODDI32
 - MREPT2020AD18_YHS_NODDI32
 - MREPT2020AD19_PKS_NODDI32
 - MREPT2020AD20_LSK_NODDI32
 - MREPT2020AD21_CBS_NODDI32
 - MREPT2020AD22_KYH_NODDI32
 - MREPT2020AD23_LSH_NODDI32
 - MREPT2020AD24_YYS_NODDI32
 - MREPT2020AD25_YSJ_NODDI32
 - MREPT2020AD26_HMH_NODDI32
 - MREPT2020AD27_YJJ_NODDI32
 - MREPT2020AD28_YYL_NODDI32
30 ['HFC2020AD01_KYM_NODDI32', 'HFC2020AD02_KS_NODDI32', 'MRE

In [12]:
dem21 = migrate_alps_subjects(
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2021",
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
    , use_tqdm=True
    , tqdm_level=1
)
print(dem21["subjects_ok"], dem21["moved_names"])

Migrating subjects: 100%|████████████████████████████████████████████████████████████████| 2/2 [00:10<00:00,  5.33s/it]


[SUMMARY] Success: 2, Fail: 0, Sum n: 2
[MOVED COUNT] 2
[MOVED NAMES]
 - HFC2021AD01_YMB_NODDI32
 - HFC2021AD02_SHS_NODDI32
2 ['HFC2021AD01_YMB_NODDI32', 'HFC2021AD02_SHS_NODDI32']


In [13]:
dem22 = migrate_alps_subjects(
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2022",
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
    , use_tqdm=True
    , tqdm_level=1
)
print(dem22["subjects_ok"], dem22["moved_names"])

Migrating subjects: 100%|██████████████████████████████████████████████████████████████| 74/74 [06:36<00:00,  5.36s/it]


[SUMMARY] Success: 74, Fail: 0, Sum n: 74
[MOVED COUNT] 74
[MOVED NAMES]
 - EPT2022AD01_KDKM_NODDI32
 - EPT2022AD02_SKM_NODDI32
 - EPT2022AD03_KKS_NODDI32
 - EPT2022AD04_KSS_NODDI32
 - EPT2022AD05_KIJD_NODDI32
 - EPT2022AD063_BPB_NODDI32
 - EPT2022AD064_LYS_NODDI32
 - EPT2022AD065_CGS_NODDI32
 - EPT2022AD066_AIS_NODDI32
 - EPT2022AD067_JSO_NODDI32
 - EPT2022AD068_NSE_NODDI32
 - EPT2022AD069_LBD_NODDI32
 - EPT2022AD06_JOJ_NODDI32
 - EPT2022AD070_KSH_NODDI32
 - EPT2022AD071_KSJ_NODDI32
 - EPT2022AD072_SDY_NODDI32
 - EPT2022AD073_KHR_NODDI32
 - EPT2022AD074_LGR_NODDI32
 - EPT2022AD07_YMY_NODDI32
 - EPT2022AD08_JIL_NODDI32
 - EPT2022AD09_JYS_NODDI32
 - EPT2022AD10_HSY_NODDI32
 - EPT2022AD11_PSI_NODDI32
 - EPT2022AD12_SYS_NODDI32
 - EPT2022AD13_SYJ_NODDI32
 - EPT2022AD14_PYS_NODDI32
 - EPT2022AD15_YSH_NODDI32
 - EPT2022AD16_CKJ_NODDI32
 - EPT2022AD17_KHS_NODDI32
 - EPT2022AD18_PKW_NODDI32
 - EPT2022AD19_YJH_NODDI32
 - EPT2022AD20_KYS_NODDI32
 - EPT2022AD21_KOJ_NODDI32
 - EPT2022AD22_NSC_NO

In [14]:
dem23 = migrate_alps_subjects(
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2023",
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
    , use_tqdm=True
    , tqdm_level=1
)
print(dem23["subjects_ok"], dem23["moved_names"])

Migrating subjects: 100%|██████████████████████████████████████████████████████████████| 22/22 [02:03<00:00,  5.62s/it]


[SUMMARY] Success: 22, Fail: 0, Sum n: 22
[MOVED COUNT] 22
[MOVED NAMES]
 - EPT2023AD01_KMH_NODDI32
 - EPT2023AD02_HUI_NODDI32
 - EPT2023AD03_MSJ_NODDI32
 - EPT2023AD04_KJU_NODDI32
 - EPT2023AD05_JIS_NODDI32
 - EPT2023AD06_LSJ_NODDI32
 - EPT2023AD07_WGJ_NODDI32
 - EPT2023AD08_KMS_NODDI32
 - EPT2023AD09_LSU_NODDI32
 - EPT2023AD10_JBY_NODDI32
 - EPT2023AD11_HSY_NODDI32
 - EPT2023AD12_YJS_NODDI32
 - EPT2023AD13_WOB_NODDI32
 - EPT2023AD14_LMS_NODDI32
 - EPT2023AD15_LSJ_NODDI32
 - EPT2023AD16_JHG_NODDI32
 - EPT2023AD17_LHG_NODDI32
 - EPT2023AD18_LMS_NODDI32
 - EPT2023AD19_KTG_NODDI32
 - EPT2023AD20_CGJ_NODDI32
 - EPT2023AD21_PDB_NODDI32
 - EPT2023AD22_KID_NODDI32
22 ['EPT2023AD01_KMH_NODDI32', 'EPT2023AD02_HUI_NODDI32', 'EPT2023AD03_MSJ_NODDI32', 'EPT2023AD04_KJU_NODDI32', 'EPT2023AD05_JIS_NODDI32', 'EPT2023AD06_LSJ_NODDI32', 'EPT2023AD07_WGJ_NODDI32', 'EPT2023AD08_KMS_NODDI32', 'EPT2023AD09_LSU_NODDI32', 'EPT2023AD10_JBY_NODDI32', 'EPT2023AD11_HSY_NODDI32', 'EPT2023AD12_YJS_NODDI32', 'EPT

In [15]:
dem24 = migrate_alps_subjects(
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\2_Dementia\EPT2024",
    r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD"
    , use_tqdm=True
    , tqdm_level=1
)
print(dem24["subjects_ok"], dem24["moved_names"])

Migrating subjects: 100%|██████████████████████████████████████████████████████████████| 13/13 [01:09<00:00,  5.32s/it]


[SUMMARY] Success: 13, Fail: 0, Sum n: 13
[MOVED COUNT] 13
[MOVED NAMES]
 - EPT2024AD01_KJI_NODDI32
 - EPT2024AD02_HUY_NODDI32
 - EPT2024AD03_NGS_NODDI32
 - EPT2024AD04_LDS_NODDI32
 - EPT2024AD05_KEJ_NODDI32
 - EPT2024AD06_PYG_NODDI32
 - EPT2024AD07_JGJ_NODDI32
 - EPT2024AD08_LMS_NODDI32
 - EPT2024AD09_CSO_NODDI32
 - EPT2024AD10_KCJ_NODDI32
 - EPT2024AD11_KHJ_NODDI32
 - EPT2024AD12_LDS_NODDI32
 - EPT2024AD13_LTO_NODDI32
13 ['EPT2024AD01_KJI_NODDI32', 'EPT2024AD02_HUY_NODDI32', 'EPT2024AD03_NGS_NODDI32', 'EPT2024AD04_LDS_NODDI32', 'EPT2024AD05_KEJ_NODDI32', 'EPT2024AD06_PYG_NODDI32', 'EPT2024AD07_JGJ_NODDI32', 'EPT2024AD08_LMS_NODDI32', 'EPT2024AD09_CSO_NODDI32', 'EPT2024AD10_KCJ_NODDI32', 'EPT2024AD11_KHJ_NODDI32', 'EPT2024AD12_LDS_NODDI32', 'EPT2024AD13_LTO_NODDI32']


In [17]:
def audit_wCvec(
    dest_root=r"D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD",
    name_contains="wCvec",
    print_missing_names=True,
):
    DEST = Path(dest_root)
    if not DEST.exists():
        raise FileNotFoundError(f"[ERROR] destination not found: {DEST}")

    subjects = [p for p in DEST.iterdir() if p.is_dir()]
    n_subjects = len(subjects)

    missing_cti = []        # No 'CTI' folder
    zero_wCvec = []         # CTI: O / wCvec*: None
    per_subject_counts = {} # {subject_name: count}
    total_files = 0

    for subj in subjects:
        cti = subj / "Results" / "CTI"
        if not cti.exists() or not cti.is_dir():
            per_subject_counts[subj.name] = 0
            missing_cti.append(subj.name)
            continue

        count = 0
        for f in cti.rglob("*"):
            if f.is_file() and (name_contains.lower() in f.name.lower()):
                count += 1
        per_subject_counts[subj.name] = count
        total_files += count
        if count == 0:
            zero_wCvec.append(subj.name)

    print(f"[DEST] {DEST}")
    print(f"[SUBJECTS] total: {n_subjects}")
    print(f"[CTI MISSING] subjects: {len(missing_cti)}")
    print(f"[wCvec FILES] total across all subjects: {total_files}")
    print(f"[SUBJECTS WITH 0 '{name_contains}']: {len(zero_wCvec)}")

    if print_missing_names:
        if missing_cti:
            print("\n[NO CTI FOLDER]")
            for name in missing_cti:
                print(f" - {name}")
        if zero_wCvec:
            print(f"\n[ZERO '{name_contains}' FILES]")
            for name in zero_wCvec:
                print(f" - {name}")

    return {
        "dest": str(DEST),
        "subjects_total": n_subjects,
        "missing_cti_subjects": missing_cti,         # CTI 폴더가 아예 없는 인원
        "zero_wCvec_subjects": zero_wCvec,           # CTI는 있으나 wCvec 파일 0개
        "per_subject_counts": per_subject_counts,    # 각 인원의 wCvec 파일 수
        "wCvec_total_files": total_files,            # 전체 wCvec 파일 총합
    }

In [18]:
res = audit_wCvec()
# res["wCvec_total_files"], res["zero_wCvec_subjects"][:5]

[DEST] D:\JHAhn\8641_conductivity\DTI_ALPS\1028\8641_Dem+MCI+AD
[SUBJECTS] total: 269
[CTI MISSING] subjects: 0
[wCvec FILES] total across all subjects: 262
[SUBJECTS WITH 0 'wCvec']: 7

[ZERO 'wCvec' FILES]
 - EPT2022AD04_KSS_NODDI32
 - EPT2022AD28_GMS_NODDI32v
 - HFC2020AD01_KYM_NODDI32
 - MREPT2020AD09_PJJ_NODDI32
 - MREPT2020AD16_CKL_NODDI32
 - MREPT2020AD17_LSH_NODDI32
 - MREPT2020AD21_CBS_NODDI32
